In [2]:
import os
from fitparse import FitFile
import pandas as pd
from sqlalchemy import create_engine, text 


import datetime as dt
from pathlib import Path


In [17]:
# Reading in the sqlite engine 
from core.functions import engine

## Adding .fit files to specific bike tables in habit db

In [11]:
# Dynamically finding the bike_files folder

for x in range(5):
    # Use .parents to go up a level in the file path, x is a number list how far back we want to go 0 = parent, 1 = grandparent, and so on
    candidate_path = Path.cwd().parents[x]
    for file in os.listdir(candidate_path):
        if file == "bike_files":
            bike_files_path = os.path.join(candidate_path, file)
            break
    break

### Function to update 'bike_workout' table

In [12]:


## Defining server upload time once to use in multiple places
today_upload_date = dt.date.today().strftime("%m-%d-%Y")

### Bike Workout table
def bike_workout_upload(raw_bike_data, today_upload_date):

    # Converting timestamp to utc datetime format for sqlite db
    raw_bike_data["timestamp"] = raw_bike_data["timestamp"].astype(str) + ".000000"

    # Grabbing min and max times for the bike workout
    # timestamp from .fit file is already in UTC
    max_time = max(raw_bike_data["timestamp"])
    min_time = min(raw_bike_data["timestamp"])


    with engine.begin() as conn:
        # Bike Workout table
        result = conn.execute(text("""INSERT INTO bike_workout (start_time, end_time, upload_date)
                            VALUES (:start_time, :end_time, :upload_date)"""),
                            {"start_time": str(min_time), "end_time": str(max_time), "upload_date": str(today_upload_date) })
        
        workout_id = result.lastrowid  # Get the ID of the workout we just inserted

    return workout_id

### Function to update 'bike_units' table

Mappings is something I should go back and dynamcially fill

In [ ]:
def bike_units_upload(raw_bike_data):

    # Data cleaning step to conver the data to long format

    ## Statically Defining the metric-to-unit mappings
    metric_unit_map = {
        'altitude': 'meters',
        'cadence': 'revolutions_per_minute',
        'distance': 'meters',
        'enhanced_altitude': 'meters',
        'enhanced_speed': 'meters_per_second',
        'position_lat': 'degrees',
        'position_long': 'degrees',
        'power': 'watts',
        'speed': 'meters_per_second'
    }

    ## Statically defining the unit to abbreviation mappings
    unit_abv_map = {
        'meters': 'm',
        'revolutions_per_minute': 'rpm',
        'meters_per_second': 'm/s',
        'degrees': 'deg',
        'watts': 'W',
    }

    ## Melt the dataframe
    bike_data_long = pd.melt(raw_bike_data, 
                id_vars=['timestamp'], 
                value_vars=['altitude', 'cadence', 'distance', 'enhanced_altitude',
                            'enhanced_speed', 'position_lat', 'position_long',
                            'power', 'speed'], 
                var_name="metric", 
                value_name="value")


    ## Add unit columns using map
    bike_data_long['unit_name'] = bike_data_long['metric'].map(metric_unit_map) 
    bike_data_long['metric_unit'] = bike_data_long['unit_name'].map(unit_abv_map)


    # Uploading data to the bike units df only if it's new 

    ## Filtering bike_data_long to only revlavent columns associated with bike_units table
    bike_units_df = bike_data_long[["metric", "unit_name", "metric_unit"]].drop_duplicates() # drop_dups to get unique values

    for _, row in bike_units_df.iterrows():
        metric_type = row["metric"]
        metric_unit = row["metric_unit"]
        unit_name = row["unit_name"]

        with engine.begin() as conn:
            ## Insert only if it doesn't already exist
            conn.execute(text("""
                INSERT OR IGNORE INTO bike_units (metric_type, metric_unit, unit_name)
                VALUES (:metric_type, :metric_unit, :unit_name)"""), 
                {"metric_type": metric_type, "metric_unit": metric_unit, "unit_name": unit_name})
            
    return bike_data_long

### Function to update 'bike_metrics' table

In [ ]:
def bike_metrics_upload(bike_data_long, workout_id):

   # Selecting data only relavent to bike_metrics table insertion 
   bike_metrics_df = bike_data_long[["timestamp", "metric", "value"]].drop_duplicates().copy()

   # workout_id comes from above insertion code
   bike_workout_id = workout_id

   # Grabbing all the metric types from the metrics table which I'll use to filter incoming bike data and insert into bike_metrics
   with engine.begin() as conn:
      bike_units = pd.read_sql_query(text("""SELECT id, metric_type 
                                          FROM bike_units"""), conn)

   bike_metrics_df["workout_id"] = bike_workout_id

   # Looping through each bike metric and uploading data from new bike file with the appropriate bike metric id 
   for _, row in bike_units.iterrows():
      metric_name = row["metric_type"]
      
      bike_metrics_upload_df = bike_metrics_df[bike_metrics_df["metric"] == metric_name].copy()

      # Now that we're filtered by metric name we can just insert the id for that metric
      bike_metrics_upload_df["metric_id"] = row["id"]
      
      # After filtering by metric column droppping it because now we have it's ID!
      bike_metrics_upload_df.drop(columns=["metric"], inplace=True)

      # Appending the new raw data to the apple_data_raw table
      bike_metrics_upload_df.to_sql(name="bike_metrics", con=engine, if_exists='append', index=False)

### Reading in new fit files and applying uploads to each table

In [ ]:
# Processing all new fit files (that don't have date already attached) and inserting appropriate data into each table via above functions

# Initializing file number counter to start at the first file
file_number = 0
total_unprocessed_files = len([file for file in os.listdir(bike_files_path) if not file.startswith(".") and "Bike Workout" not in file])

if total_unprocessed_files != 0:
    for file_path in os.listdir(bike_files_path):
        # Handling my .gitkeep file and similar files 
        if file_path.startswith("."):
            continue
        # Filtering to fit files that I have not processed yet (All processed files get renamed)
        if "Bike Workout" not in file_path:
            bike_fit_file = os.path.join(bike_files_path, file_path)

            # This is where python creates the connection to the fit file and reads it in
            fitfile = FitFile(bike_fit_file)

            # Initializing rows
            rows = []

            # Iterate over all record messages
            for record in fitfile.get_messages('record'):
                row = {}

                # Each record contains multiple data fields
                for field in record:
                    row[field.name] = field.value

                rows.append(row)

            # Create DataFrame
            df = pd.DataFrame(rows)

            # Dropping heart rate because it's blank
            raw_bike_data = df.drop(columns="heart_rate")

            # Putting bike data into appropriate tables
            ## Inserting data into the Bike workouts table
            workout_id = bike_workout_upload(raw_bike_data, today_upload_date)

            ## Inserting data into the Bike units table
            bike_data_long = bike_units_upload(raw_bike_data)    

            ## Inserting data into the Bike metrics table
            bike_metrics_upload(bike_data_long, workout_id)

            # For each fit file changing the name so I don't read it in again  
            for record in fitfile.get_messages('record'):
                for field in record:
                    if field.name == "timestamp":     #, field.value, field.units
                        file_date = field.value
                        # String formatting the date variable
                        file_date = file_date.strftime('%m-%d-%Y')
                        break
                if file_date:
                    break

            # Important to discard the connection to fitfile before saving over it otherwise os.rename will error out because python is still using the file
            del fitfile

            # This would be bad if we can't find timestamp data
            if not file_date:
                print(f"No timestamp found for {file_path}")
                continue
            
            # Renaming current file to fit naming schema: 01-14-2026 Bike Workout
            os.rename(bike_fit_file, os.path.join(bike_files_path, f"{file_date} Bike Workout.fit"))

            file_number += 1
            
            # Readable step to ensure correct number of files are processed
            print(f"Processed file '{file_path}', number {file_number} out of {total_unprocessed_files}")

print("no files to upload ")

Processed file 'vjklvjklvdfkldfv.fit', number 1 out of 1
no files to upload 


## Using newly inserted bike data to appened apple_workouts for cycling workouts

In [21]:
# Grabbing dates of all bike workouts from apple

with engine.begin() as conn:
    apple_cycle = pd.read_sql_query(
        text("""
            SELECT StartDate, activity, workout_id 
            FROM apple_workouts
            where activity = 'Cycling' and metric = 'Duration'
            order by StartDate desc
        """), conn,
        parse_dates=["StartDate"],
        dtype={"workout_id": "Int64"})

    # Localizing to UTC timezone for merging later
    apple_cycle["StartDate"] = apple_cycle["StartDate"].dt.tz_localize("UTC")

    # Abstracting to date for merge to .fit data (won't merge perfectly on datetime)
    apple_cycle["apple_date"] =  apple_cycle["StartDate"].dt.date

In [22]:
# Grabbing total distance and average wattage from .fit files bike workouts we just uploaded and combining

with engine.begin() as conn:
    fit_distance = pd.read_sql_query(
        text("""
            SELECT bw.id, bw.start_time, MAX(bm.value) as 'max_distance'
            from bike_workout bw 
            LEFT OUTER JOIN bike_metrics bm 
            on bm.workout_id = bw.id 
            LEFT OUTER JOIN bike_units bu 
            on bu.id = bm.metric_id
            where bu.metric_type = 'distance' AND bw.upload_date = :today_upload_date 
            GROUP BY bw.id
        """), conn,
        params={"today_upload_date": str(today_upload_date)},
        parse_dates=["start_time"])


    fit_watts = pd.read_sql_query(
        text("""
        SELECT bw.id, ROUND(AVG(bm.value), 2) as 'avg_watts'
        from bike_workout bw 
        LEFT OUTER JOIN bike_metrics bm 
        on bm.workout_id = bw.id 
        LEFT OUTER JOIN bike_units bu 
        on bu.id = bm.metric_id
        where bu.metric_type = 'power' AND bw.upload_date = :today_upload_date
        GROUP BY bw.id
        """), conn,
        params={"today_upload_date": str(today_upload_date)},
        parse_dates=["start_time"])

# Localizing to UTC timezone for merging because this is the start time var that will stay
fit_distance["start_time"] = fit_distance["start_time"].dt.tz_localize("UTC")

fit_distance["fit_date"] = fit_distance["start_time"].dt.date

# Merging various calculations of bike metric data together at bike workout level into singular df for upload
fit_total = pd.merge(fit_distance, fit_watts, on="id", how="left")

For this below code to work there needs to be associated indoor cycling workouts in the apple workouts df already 

In [ ]:
# Lopping through each variable option to upload those sepcific options to apple workouts table

# Using fit file data to find corresponding apple bike workouts 
combined = pd.merge(fit_total, apple_cycle, left_on="fit_date", right_on="apple_date", how="left")

# Meters to miles conversion
combined["max_distance_miles"] = (combined["max_distance"] / 1609).round(2)

# Converting start date to sqlite appropriate format
combined["StartDate"] = combined["StartDate"].dt.strftime('%Y-%m-%d %H:%M:%S') + ".000000"

# Dropping unnecessary/helper columns
combined.drop(columns=["start_time", "max_distance", "fit_date", "apple_date"], inplace=True)

## Defining dict with variable options   
variables = {"max_distance_miles": {"metric":"DistanceCycling", "measurement_type": "sum"}, 
            "avg_watts": {"metric":"AvgWatts", "measurement_type": "average"}}

with engine.begin() as conn:
    for var_name, var_config in variables.items() :
        temp_df = combined[combined[var_name].notnull()][["StartDate",  "activity", "workout_id", var_name]]

        temp_df["metric"] = var_config["metric"]
        temp_df["measurement_type"] = var_config["measurement_type"]
        temp_df["activity_type"] = "Cardio"

        for _, row in temp_df.iterrows():

            conn.execute(text("""
                    INSERT INTO apple_workouts (StartDate, activity, metric, measurement_type, value, d_unit, activity_type, workout_id)
                    VALUES (:StartDate, :activity, :metric, :measurement_type, :value, :d_unit, :activity_type, :workout_id)"""), {
                    'StartDate': str(row["StartDate"]),
                    'activity': row["activity"],
                    'metric': row["metric"],
                    'measurement_type': row["measurement_type"],
                    'value': row[var_name],  # Taking the variable name from the dict key
                    'd_unit': None,
                    'activity_type': row["activity_type"],
                    'workout_id': row["workout_id"]
                })